# selenium을 이용한 데이터 크롤링

In [ ]:
from selenium import webdriver as wd

In [ ]:
# 자동제어 브라우저
driver = wd.Chrome()

In [23]:
# 타겟 사이트 진입
driver.get('https://www.opinet.co.kr/searRgSelect.do')

- 시도 정보 추출
- css selector
  ```
    - 시도 정보를 가진 <option> 태그를 모두 찾는 ess selector -> 18개 검색, 이 중 첫번째는 데이터 x(프럼프트 내용)
    - SIDO_NM0 > option
  ```

In [24]:
# css celector 사용한다는 지정 -> By, 표현식
from selenium.webdriver.common.by import By

# HTML문서에서 id 속성은 유일한 값을 가지고(상호간의 약속, 필수x) -> 이를 특정
# css selector 표현 => #id 값 => 직관적/기계적 찾을 때 1순위 체크 포인트
# css selector 표현 => 직계자식(바로 밑에 존재하는 하위 요소) => >
# css selector 표현 => 하위모든자식(바로 밑에 존재하는 모든 하위 요소) => <- 공백 한칸
sidos = driver.find_elements(By.CSS_SELECTOR, '#SIDO_NM0 > option')

In [25]:
# 18개 탐색
len(sidos)

0

In [ ]:
# 반복적 탐색 -> 모든 <option> 태그에서 선택 `값(value)`을 미리 다 추출함
for option in sidos[:2]: # WebElement가 탐색됨
    # DP되는 텍스트 -> 원하는 데이터 x
    # print(option.text)
    # 원하는 데이터는 <option>의 속성 value의 값
    print(f"[option.get_attribute('value').strip()]") # 샘플값 체크
    # 첫번째 데이터는 존재 x => WHY => SELECT 태그의 프럼프트 담당 => 데이터 x => 제거 필요

    # sidos_value 리스트를 구하시오, 멤버수는 17개(실제 시도 데이터만 추출)
    [sidos_value for sidos_value in option.get_attribute('value').strip() ]

In [55]:
 # sidos_value 리스트를 구하시오, 멤버수는 17개(실제 시도 데이터만 추출)
sidos_value = [option.get_attribute('value').strip() for option in sidos[1:]]

sidos_value

[]

#### 요구사항

- 전체 시도/시군구 별 데이터 획득하는 flow(행위) 표현
  - 시도를 반복하면서(ajax, 백그라운드 비동기 통신 진행 -> 화면 그대로 -> 뒤로 통신 -> 부분 화면 갱신) > 시군구 선택(깜빡임:리로드, form 전송) > 잠시 대기(보수적) > 로딩 완료 > 엑셀 다운 > 잠시 대기(보수적) > next (다음 시군구)

In [ ]:
import time

# 1.시도를 순회하면서 화면의 변화 체크 -> 실습상 서울만 진행 -> 추후 전국 시도로 확장 
for sido_v in sidos_value: #[:2]:
    print(sido_v)
    # 시도가 변경되면 ajax가 작동하면 화면이 변경된다
    # 사용자는 시도를 선택(변경) 해야함(액션)
    # 선택(변경)은 입력받는 태그(input, select, radio, checkbox, textarea)에 값을 설정하는 행위임
    # 시도를 대변하는 select 태그를 특정(css selector) -> 입력 행위 수행(send_key(값))
    driver.find_element(By.ID, "SIDO_NM0").send_keys(sido_v)
    # 잠시 대기(통신, 화면갱신) -> 명시적-> time.sleep(5)/암묵적(특정 조건이 만족될 때까지 계속 체크)
    time.sleep(2)
    # 화면 변경됨
    #break # 서울만 진행

서울특별시
부산광역시
대구광역시
인천광역시
광주광역시
대전광역시
울산광역시
세종특별자치시
경기도
강원특별자치도
충청북도
충청남도
전북특별자치도
전라남도
경상북도
경상남도
제주특별자치도


In [ ]:
import time

# 1.시도를 순회하면서 화면의 변화 체크 -> 실습상 서울만 진행 -> 추후 전국 시도로 확장 
for sido_v in sidos_value: #[:2]:
# 2. 특정 시도 선택
    print(sido_v)
    # <select>를 특정해서 사용자 선택값을 서울특별시(side_v)로 세팅(send_key())하라
    driver.find_element(By.ID, "SIDO_NM0").send_keys(sido_v)
    # 2초 강제 대기 -> ajax 통신 후 화면에 변화가 올 때까지 임의 대기시킴(다른 코드 전개x )
    # 2초의 근거 -> 현재 환경에서 테스트한 결과 2초 이내면 안전하게 전화처리가 되었다 -> 환경이 바뀌면 다시 구성
    # 명시적 대기 -> 리뷰시 암묵적 대기 코드로 전환 연습
    time.sleep(2)

# 3.특정 시도의 시군구의 값을 획득 -> 시군구를 선택하면 -> form 전송 발생 
# -> Dom tree 새로 구성 -> 메모리에 객체 주소 모두 리세팅 -> 기존에 참조한 객체가 모두 소멸
# -> 기존값 사용 => 에러 발생 -> 진행 중단
# <요구사항>
# 시군구를 특정(#SIGUNGU_NM0)하여 시군구 값(강남구, 강동구..중량구) 문자열을 리스트로 구하시오

    sigungu_value = [
        option.get_attribute('value').strip() 
        for option in driver.find_elements(By.CSS_SELECTOR, '#SIGUNGU_NM0 > option')[1:]]
    print(sigungu_value)
    for sigungu_v in sigungu_value:
        # 4-1. 시군구 선택 태그(<select>)를 특정하여 -> form 전송 발생 -> 객체주소 변경/소멸 -> 값 세팅
        driver.find_element(By.ID,"SIGUNGU_NM0").send_keys(sigungu_v)
        # 4-2. 명시적 잠시 대기
        time.sleep(2.5)
        # 4-3. 엑셀 파일 다운로드 -> 엑셀 다운 버튼 특정(css selector), 클릭
        driver.find_element(By.CSS_SELECTOR,'.btn_type6_ex_save').click()
        # 4-4.명시적 대기(파일 다운로드 -> 로컬 PC에 저장까지 시간)
        time.sleep(10)

    break # 서울만 진행

서울특별시
['강남구', '강동구', '강북구', '강서구', '관악구', '광진구', '구로구', '금천구', '노원구', '도봉구', '동대문구', '동작구', '마포구', '서대문구', '서초구', '성동구', '성북구', '송파구', '양천구', '영등포구', '용산구', '은평구', '종로구', '중구', '중랑구']


In [33]:
# 브라우저 닫기
# driver.close() # 현재 탭 하나만 닫는다 -> 프로세스가 살아있음 -> 언젠가 os가 먹통됨
driver.quit() # 모든 탭을 답고, WebDreiver 세션을 완전 종료

# Transform : n개 엑셀 => DataFrame 처리

In [32]:
# 특정 폴더에서 특정 파일(패턴)을 모두 가져온다(집어낸다/잡는다)
import glob

# 경로 구분자 => \\ or /
files = glob.glob('C:/Users/Dell3571/Downloads/*xls')
files[:2]

['C:/Users/Dell3571/Downloads\\지역_위치별(주유소) (10).xls',
 'C:/Users/Dell3571/Downloads\\지역_위치별(주유소) (11).xls']

In [35]:
!pip install xlrd


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [43]:
import pandas as pd
# xls => DataFrame n개 구성하여 list에 담는다. 단, 순서 x
oils = list()
for file in files:
    # 내용 체크 -> 상위 2개 행은 데이터 x -> 컬럼은 2번(0부터 출발)행부터 진행
    df = pd.read_excel(file, header = 2)
    oils.append(df)
    # break
len(oils)

26

In [44]:
# 샘플 데이터 1개 체크
df.sample(2) # 무작위로 2개 체크

,지역,상호,주소,상표,전화번호,셀프여부,고급휘발유,휘발유,경유,실내등유
6,서울특별시,구일주유소,서울 구로구 구일로 94 (구로동),HD현대오일뱅크,02-868-5758,Y,2034,1894,1934,-
5,서울특별시,구인주유소,서울 구로구 경인로 558 (구로동),SK에너지,02-2677-5010,Y,-,1894,1934,-


In [45]:
# oils라는 리스트에 모든 서울시 주유소 가격 데이터가 df로 모여있음 -> 1개의 df로 구성 
# -> 병합(`concat():n개 -> 단순 합치기`, merge():2개->join 컨셉)
# axis는 병합(단순 합치기의 방향성(좌표축)) 
# axis : 체크필요 : 0행(컬럼 확장, 데이터 수는 최대값 고정), 1열(컬럼 고정, 데이터 수 확장)
final_df = pd.concat(oils, axis=0)
# 데이터의 형상 => (1차원 개수, 2차원 개수)
final_df.shape

(423, 10)

In [46]:
final_df.sample(10)

,지역,상호,주소,상표,전화번호,셀프여부,고급휘발유,휘발유,경유,실내등유
13,서울특별시,국회대로주유소,서울 영등포구 국회대로 746 (여의도동),HD현대오일뱅크,02-761-5122,Y,2197,1917,1967,-
12,서울특별시,범아주유소,서울 중랑구 동일로 881,S-OIL,02-974-8356,N,-,1996,1998,1700
12,서울특별시,백제고분로주유소,서울 송파구 오금로 143 (방이동),SK에너지,02-2202-4050,Y,2068,1888,1898,-
16,서울특별시,(주)정직한주유소,서울 송파구 오금로 455 (거여동),HD현대오일뱅크,02-406-4634,Y,2052,1902,1952,-
3,서울특별시,현대오일뱅크(주)직영 양천셀프주유소,서울 양천구 목동로 17,HD현대오일뱅크,02-2644-5185,Y,1995,1817,1810,-
11,서울특별시,씨앤에스유통(주) 마곡드림주유소,서울 강서구 방화대로 254(마곡동),S-OIL,02-2667-0113,Y,-,1905,1995,-
4,서울특별시,덕릉로주유소,서울 강북구 덕릉로 158 (번동),S-OIL,02-989-9806,Y,-,1899,1860,-
7,서울특별시,(주)정건에너지 배봉로지점,서울 동대문구 서울시립대로 108,HD현대오일뱅크,02-2242-0556,Y,-,1899,1860,1800
19,서울특별시,반포그린주유소,서울 서초구 서초중앙로 239,GS칼텍스,02-532-5147,Y,2095,1935,1975,-
4,서울특별시,HD현대오일뱅크(주)직영 소월길주유소,서울 용산구 소월로 66 (후암동),HD현대오일뱅크,02-318-3314,N,2368,1898,1820,-


In [ ]:
# 임시 csv or xls에 최종 결과물 저장
# 파일명 oil_20260311.csv -> s3에 업로드
final_df.to_csv('oil_20260311.csv') # 시간을 자동으로 계산해서 세팅되게 구성(업그레이드) 개선(모르겠다...)

- 사용된 xls 파일은 모두 삭제 -> os 모듈에서 파일 삭제 함수 사용 (리뷰 때 체크)
- 저장되는 공간이 깨끗해짐 -> 다음 스케줄 작동 시 문제 발생 안됨(당일 데이터는 당일 처리 기본)

In [ ]:
# xls 파일 삭제하기
import os
for f in files:
    os.remove(f)

# Load(생략)